# CNN-EXP-034 — SAFE HOLDOUT EVALUATION

Этот notebook **НЕ обучает модель заново**.

Он предназначен именно для ситуации, когда после успешных 10,000 training steps
kernel упал на full-holdout evaluation.

Загружается уже сохранённый checkpoint:

```text
artifacts/experiments/
CNN_EXP034_DUALPATH_RF2053_PERBASE_POISSON/
run_001/checkpoint_step_010000.pt
```

## Почему старый evaluator мог убить kernel

Старый код накапливал в RAM все per-base predictions и counts для примерно
десятков миллионов position-strand observations, а затем `sklearn` делал
глобальную сортировку для Average Precision.

Это создаёт несколько больших временных копий и `int64` sort indices.

Здесь вместо этого:

- **все negatives не сохраняются**;
- AP считается streaming histogram'ом;
- positive scores сохраняются полностью (их мало) для точного Spearman;
- 32-bp regional diagnostics сохраняются полностью — они уже достаточно маленькие.

Получаем:

1. `per_base_ap_hist_1m` — очень точная histogram approximation raw AP;
2. `per_base_ap_sigmoid_5dp` — exact AP после `sigmoid + round(5 decimals)`;
3. raw positive-only dense-rank Spearman;
4. quantized sigmoid-5dp dense-rank Spearman;
5. прежние 32-bp diagnostics для сравнения с EXP033.


In [1]:
import numpy as np

a = np.zeros(1_000_001, dtype=np.int64)
b = np.zeros(1_000_001, dtype=np.int64)

print(a.nbytes / 1024**2)
print(b.nbytes / 1024**2)
print("OK")

7.629402160644531
7.629402160644531
OK


In [2]:
from pathlib import Path
import gc
import json
import math
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_directml

from sklearn.metrics import average_precision_score
from scipy.stats import pearsonr, rankdata

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (
        CURRENT_DIR,
        *CURRENT_DIR.parents,
    )
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    ProfileWindowConfig,
    ProfileWindowGenerator,
    ResidualBlock,
    create_cnn_presence_intensity,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [3]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [4]:
# ============================================================
# DATA / CONFIG — identical to EXP034
# ============================================================

split = load_split(PROJECT_ROOT)

sequences, fasta_alphabet = load_baseline_sequences(split)

TARGET_LENGTH = 2048
CONTEXT_FLANK = 1026
INPUT_LENGTH = TARGET_LENGTH + 2 * CONTEXT_FLANK
REGION_WIDTH = 32

BLOCK_SETTINGS_RF2053 = (
    (7, 1),
    (3, 2),
    (3, 4),
    (3, 8),
    (3, 16),
    (3, 32),
    (3, 64),
    (3, 128),
    (3, 256),
)

CONTEXT_RF_BP = (
    1
    + sum(
        2 * (kernel_size - 1) * dilation
        for kernel_size, dilation
        in BLOCK_SETTINGS_RF2053
    )
)

assert CONTEXT_RF_BP == 2053

profile_config = ProfileWindowConfig(
    target_length=TARGET_LENGTH,
    context_flank=CONTEXT_FLANK,
    positive_branch_fraction=0.5,
)

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

validation_tiles = validation_generator.tile_index.copy()

print("Tiles:", len(validation_tiles))
print(
    "Position-strand:",
    int(validation_tiles["allowed_position_strand"].sum()),
)
print(
    "Positives:",
    int(validation_tiles["positive_positions"].sum()),
)
print(
    "Contigs:",
    validation_tiles["contig"].nunique(),
)


Tiles: 24700
Position-strand: 83807486
Positives: 56205
Contigs: 3


In [5]:
# ============================================================
# MODEL — identical to EXP034
# ============================================================

class LocalMotifBranch(nn.Module):
    def __init__(
        self,
        input_channels=5,
        channels=32,
    ):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(
                input_channels,
                channels,
                kernel_size=11,
                padding=5,
            ),
            nn.BatchNorm1d(channels),
            nn.GELU(),
        )

        self.block1 = ResidualBlock(
            channels=channels,
            kernel_size=5,
            dilation=1,
        )

        self.block2 = ResidualBlock(
            channels=channels,
            kernel_size=5,
            dilation=2,
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        return x


class DualPathInitiationModel(nn.Module):
    def __init__(
        self,
        input_channels=5,
        context_channels=32,
        local_channels=32,
        fusion_channels=48,
    ):
        super().__init__()

        context_model = create_cnn_presence_intensity(
            input_channels=input_channels,
            channels=context_channels,
            block_settings=BLOCK_SETTINGS_RF2053,
        )

        self.context_backbone = context_model.backbone

        self.local_branch = LocalMotifBranch(
            input_channels=input_channels,
            channels=local_channels,
        )

        self.fusion = nn.Sequential(
            nn.Conv1d(
                context_channels + local_channels,
                fusion_channels,
                kernel_size=1,
            ),
            nn.GELU(),
            nn.Conv1d(
                fusion_channels,
                fusion_channels,
                kernel_size=1,
            ),
            nn.GELU(),
        )

        self.rate_head = nn.Conv1d(
            fusion_channels,
            1,
            kernel_size=1,
        )

        nn.init.constant_(
            self.rate_head.bias,
            -8.0,
        )

    def forward(self, x):
        context_features = self.context_backbone(x)
        local_features = self.local_branch(x)

        fused = torch.cat(
            (
                context_features,
                local_features,
            ),
            dim=1,
        ).contiguous()

        fused = self.fusion(fused)

        return self.rate_head(fused)


def create_model():
    return DualPathInitiationModel()


def crop_target(x):
    return x[
        :,
        :,
        CONTEXT_FLANK:
        CONTEXT_FLANK + TARGET_LENGTH
    ].contiguous()


def forward_both_oriented(
    model,
    x_both,
    batch_size,
):
    all_log_rate = model(x_both)

    plus = crop_target(
        all_log_rate[:batch_size]
    )

    minus_reverse = crop_target(
        all_log_rate[batch_size:]
    )

    return plus, minus_reverse


In [6]:
# ============================================================
# LOAD SAVED 10K CHECKPOINT — NO RETRAINING
# ============================================================

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP034_DUALPATH_RF2053_PERBASE_POISSON"
    / "run_001"
)

CHECKPOINT_PATH = (
    RUN_DIR
    / "checkpoint_step_010000.pt"
)

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}"
    )

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint:", CHECKPOINT_PATH)
print("Step:", checkpoint["step"])
print("Architecture:", checkpoint["architecture"])
print("Saved EMA loss:", checkpoint.get("ema_loss"))

# validation_model = create_model()

# validation_model.load_state_dict(
#     checkpoint["model_state_dict"]
# )

# validation_model = validation_model.to(device)
# validation_model.eval()

# # Release the CPU checkpoint object after weights are loaded.
# del checkpoint
# gc.collect()

# print("Model loaded: OK")


Checkpoint: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP034_DUALPATH_RF2053_PERBASE_POISSON\run_001\checkpoint_step_010000.pt
Step: 10000
Architecture: dualpath_local35_context2053_perbase_poisson
Saved EMA loss: -36.851894635542216


In [7]:
import gc
import torch
import torch_directml

gc.collect()

print("1. Creating CPU model...")
validation_model = create_model()
print("   CPU model created")

print("2. Loading checkpoint to CPU...")
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)
print("   checkpoint loaded")

print("3. Loading state_dict...")
validation_model.load_state_dict(
    checkpoint["model_state_dict"]
)
print("   weights loaded")

del checkpoint
gc.collect()

print("4. Creating DirectML device...")
device = torch_directml.device()
print("   device:", device)

print("5. Moving model to DirectML...")
validation_model = validation_model.to(device)
print("   MODEL ON DIRECTML — OK")

print("6. Switching eval mode...")
validation_model.eval()
print("   DONE")

1. Creating CPU model...
   CPU model created
2. Loading checkpoint to CPU...
   checkpoint loaded
3. Loading state_dict...
   weights loaded
4. Creating DirectML device...
   device: privateuseone:0
5. Moving model to DirectML...
   MODEL ON DIRECTML — OK
6. Switching eval mode...
   DONE


## Про отрицательный training loss

В EXP034 Poisson loss записан без константы `log(y!)`:

```text
exp(z) - y*z
```

Это сделано намеренно: константа не влияет на gradients.

Поэтому при `y > 1` значение этого выражения **может быть отрицательным**.
EMA около `-36.85` сам по себе не означает ошибку или divergence.


In [8]:
# ============================================================
# STREAMING AP HELPERS
# ============================================================

RAW_LOG_MIN = -20.0
RAW_LOG_MAX = 15.0

# 1,000,001 bins over 35 log-rate units:
# bin width ~3.5e-5.
RAW_AP_BINS = 1_000_001

SUBMISSION_LEVELS = 100_001


def update_histogram(
    hist,
    indices,
):
    values, counts = np.unique(
        indices,
        return_counts=True,
    )
    hist[values] += counts.astype(np.int64)


def ap_from_histograms(
    positive_hist,
    total_hist,
):
    positives = int(
        positive_hist.sum()
    )

    if positives <= 0:
        return float("nan")

    # Descending score order.
    pos_desc = positive_hist[::-1]
    total_desc = total_hist[::-1]

    tp = np.cumsum(
        pos_desc,
        dtype=np.int64,
    )

    predicted_positive = np.cumsum(
        total_desc,
        dtype=np.int64,
    )

    precision = (
        tp
        / np.maximum(
            predicted_positive,
            1,
        )
    )

    # sklearn AP:
    # sum over threshold groups of delta recall * precision.
    return float(
        np.sum(
            (
                pos_desc
                / positives
            )
            * precision
        )
    )


def raw_lograte_bin(
    scores,
):
    clipped = np.clip(
        scores,
        RAW_LOG_MIN,
        RAW_LOG_MAX,
    )

    scaled = (
        (
            clipped - RAW_LOG_MIN
        )
        / (
            RAW_LOG_MAX - RAW_LOG_MIN
        )
    )

    return np.rint(
        scaled
        * (
            RAW_AP_BINS - 1
        )
    ).astype(np.int32)


def sigmoid_5dp_bin(
    scores,
):
    clipped = np.clip(
        scores,
        -30.0,
        30.0,
    )

    p = (
        1.0
        / (
            1.0
            + np.exp(
                -clipped
            )
        )
    )

    return np.rint(
        p * 100_000.0
    ).astype(np.int32)


In [9]:
# ============================================================
# ULTRA-SAFE FULL HOLDOUT — DIRECTML
# ============================================================

import gc
import time
import numpy as np
import torch

EVAL_BATCH_SIZE = 1
PROGRESS_EVERY = 25

raw_total_hist = np.zeros(
    RAW_AP_BINS,
    dtype=np.int64,
)

raw_positive_hist = np.zeros(
    RAW_AP_BINS,
    dtype=np.int64,
)

sigmoid_total_hist = np.zeros(
    SUBMISSION_LEVELS,
    dtype=np.int64,
)

sigmoid_positive_hist = np.zeros(
    SUBMISSION_LEVELS,
    dtype=np.int64,
)

positive_score_chunks = []
positive_count_chunks = []

started = time.perf_counter()
positions = 0

validation_model.eval()

with torch.no_grad():

    for batch_no, start in enumerate(
        range(
            0,
            len(validation_tiles),
            EVAL_BATCH_SIZE,
        ),
        1,
    ):

        selected = validation_tiles.iloc[
            start:
            start + EVAL_BATCH_SIZE
        ]

        batch = validation_generator.make_batch(
            selected
        )

        B = len(selected)

        # ------------------------------------
        # CPU -> DirectML
        # ------------------------------------

        x_gpu = (
            batch
            .x_both_strands
            .to(device)
        )

        # ------------------------------------
        # FORWARD
        # ------------------------------------

        (
            plus_gpu,
            minus_reverse_gpu,
        ) = forward_both_oriented(
            validation_model,
            x_gpu,
            batch_size=B,
        )

        # ------------------------------------
        # IMPORTANT:
        # copy GPU tensors to CPU FIRST.
        # NO torch.flip on DirectML.
        # ------------------------------------

        plus_score = (
            plus_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        minus_reverse_score = (
            minus_reverse_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        # reverse already on CPU
        minus_score = (
            minus_reverse_score[
                :,
                ::-1
            ]
            .copy()
        )

        # Kill DirectML tensors immediately
        del (
            x_gpu,
            plus_gpu,
            minus_reverse_gpu,
        )

        # ------------------------------------
        # CPU processing
        # ------------------------------------

        score = np.stack(
            (
                plus_score,
                minus_score,
            ),
            axis=1,
        )

        count = (
            batch
            .count
            .numpy()
            .astype(
                np.float32,
                copy=False,
            )
        )

        mask = (
            batch
            .mask
            .numpy()
            .astype(
                bool,
                copy=False,
            )
        )

        valid_score = score[
            mask
        ]

        valid_count = count[
            mask
        ]

        positive = (
            valid_count > 0
        )

        # ------------------------------------
        # RAW histogram AP
        # ------------------------------------

        raw_idx = raw_lograte_bin(
            valid_score
        )

        update_histogram(
            raw_total_hist,
            raw_idx,
        )

        if positive.any():

            update_histogram(
                raw_positive_hist,
                raw_idx[
                    positive
                ],
            )

        # ------------------------------------
        # 5-decimal sigmoid AP
        # ------------------------------------

        sigmoid_idx = sigmoid_5dp_bin(
            valid_score
        )

        update_histogram(
            sigmoid_total_hist,
            sigmoid_idx,
        )

        if positive.any():

            update_histogram(
                sigmoid_positive_hist,
                sigmoid_idx[
                    positive
                ],
            )

            positive_score_chunks.append(
                valid_score[
                    positive
                ]
                .astype(
                    np.float32,
                    copy=True,
                )
            )

            positive_count_chunks.append(
                valid_count[
                    positive
                ]
                .astype(
                    np.float32,
                    copy=True,
                )
            )

        positions += int(
            mask.sum()
        )

        # Explicitly kill batch-local CPU arrays.
        del (
            batch,
            selected,
            plus_score,
            minus_score,
            minus_reverse_score,
            score,
            count,
            mask,
            valid_score,
            valid_count,
            positive,
            raw_idx,
            sigmoid_idx,
        )

        # Every 25 iterations clean Python refs.
        if batch_no % 25 == 0:
            gc.collect()

        if (
            batch_no == 1
            or batch_no % PROGRESS_EVERY == 0
            or start + B >= len(validation_tiles)
        ):

            elapsed = (
                time.perf_counter()
                - started
            )

            print(
                f"tiles={start + B:>6}"
                f"/{len(validation_tiles)} "
                f"positions={positions:>12,} "
                f"positions/s="
                f"{positions / elapsed:,.0f}"
            )


print()
print("FULL INFERENCE FINISHED")
print()

per_base_ap_hist_1m = (
    ap_from_histograms(
        raw_positive_hist,
        raw_total_hist,
    )
)

per_base_ap_sigmoid_5dp = (
    ap_from_histograms(
        sigmoid_positive_hist,
        sigmoid_total_hist,
    )
)

positive_scores = np.concatenate(
    positive_score_chunks
)

positive_counts = np.concatenate(
    positive_count_chunks
)

raw_pred_rank = rankdata(
    positive_scores,
    method="dense",
)

target_rank = rankdata(
    positive_counts,
    method="dense",
)

epic_spearman_raw = float(
    pearsonr(
        raw_pred_rank,
        target_rank,
    )[0]
)

positive_sigmoid = (
    sigmoid_5dp_bin(
        positive_scores
    )
)

quant_pred_rank = rankdata(
    positive_sigmoid,
    method="dense",
)

epic_spearman_sigmoid_5dp = float(
    pearsonr(
        quant_pred_rank,
        target_rank,
    )[0]
)

print(
    "per-base AP raw:",
    per_base_ap_hist_1m,
)

print(
    "per-base AP sigmoid5:",
    per_base_ap_sigmoid_5dp,
)

print(
    "EPIC Spearman raw:",
    epic_spearman_raw,
)

print(
    "EPIC Spearman sigmoid5:",
    epic_spearman_sigmoid_5dp,
)

tiles=     1/24700 positions=         210 positions/s=2,304
tiles=    25/24700 positions=      66,288 positions/s=254,469
tiles=    50/24700 positions=     123,026 positions/s=281,127
tiles=    75/24700 positions=     191,634 positions/s=306,119
tiles=   100/24700 positions=     264,146 positions/s=323,381
tiles=   125/24700 positions=     312,264 positions/s=313,180
tiles=   150/24700 positions=     385,388 positions/s=328,859
tiles=   175/24700 positions=     459,706 positions/s=339,818
tiles=   200/24700 positions=     518,990 positions/s=339,052
tiles=   225/24700 positions=     599,444 positions/s=350,248
tiles=   250/24700 positions=     660,784 positions/s=350,360
tiles=   275/24700 positions=     735,408 positions/s=355,899
tiles=   300/24700 positions=     817,638 positions/s=363,317
tiles=   325/24700 positions=     896,382 positions/s=365,757
tiles=   350/24700 positions=     965,760 positions/s=367,654
tiles=   375/24700 positions=   1,029,332 positions/s=366,151
tiles=   4

In [10]:
# ============================================================
# COMPUTE METRICS FROM SMALL / STREAMED STATE
# ============================================================

per_base_ap_hist_1m = ap_from_histograms(
    raw_positive_hist,
    raw_total_hist,
)

per_base_ap_sigmoid_5dp = ap_from_histograms(
    sigmoid_positive_hist,
    sigmoid_total_hist,
)

positive_scores = np.concatenate(
    positive_score_chunks
)

positive_counts = np.concatenate(
    positive_count_chunks
)

raw_pred_rank = rankdata(
    positive_scores,
    method="dense",
)

target_rank = rankdata(
    positive_counts,
    method="dense",
)

epic_spearman_raw = float(
    pearsonr(
        raw_pred_rank,
        target_rank,
    )[0]
)

positive_sigmoid_levels = sigmoid_5dp_bin(
    positive_scores
)

quant_pred_rank = rankdata(
    positive_sigmoid_levels,
    method="dense",
)

epic_spearman_sigmoid_5dp = float(
    pearsonr(
        quant_pred_rank,
        target_rank,
    )[0]
)

regional_scores = np.concatenate(
    regional_score_chunks
)

regional_positive = np.concatenate(
    regional_positive_chunks
)

regional_tss_count = (
    np.concatenate(
        regional_tss_count_chunks
    )
    .astype(np.int64)
)

regional_ap32 = float(
    average_precision_score(
        regional_positive,
        regional_scores,
    )
)

order = np.argsort(
    -regional_scores,
    kind="mergesort",
)

total_tss = int(
    regional_tss_count.sum()
)

top10_n = int(
    math.ceil(
        0.10 * len(order)
    )
)

top10_tss_recall = float(
    regional_tss_count[
        order[:top10_n]
    ].sum()
    / total_tss
)

cumulative_tss = np.cumsum(
    regional_tss_count[
        order
    ]
)

needed_95 = int(
    np.searchsorted(
        cumulative_tss,
        0.95 * total_tss,
        side="left",
    )
    + 1
)

fraction_regions_for_95_tss = float(
    needed_95
    / len(order)
)

elapsed = (
    time.perf_counter()
    - started
)

holdout_summary = pd.DataFrame([
    {
        "experiment": "CNN-EXP-034",
        "step": 10000,
        "valid_position_strand": int(
            raw_total_hist.sum()
        ),
        "positive_positions": int(
            raw_positive_hist.sum()
        ),
        "per_base_ap_hist_1m": (
            per_base_ap_hist_1m
        ),
        "per_base_ap_sigmoid_5dp": (
            per_base_ap_sigmoid_5dp
        ),
        "epic_dense_rank_spearman_raw": (
            epic_spearman_raw
        ),
        "epic_dense_rank_spearman_sigmoid_5dp": (
            epic_spearman_sigmoid_5dp
        ),
        "regional_ap32_from_max_base_score": (
            regional_ap32
        ),
        "top10_region_tss_recall": (
            top10_tss_recall
        ),
        "fraction_regions_for_95pct_tss": (
            fraction_regions_for_95_tss
        ),
        "validation_minutes": float(
            elapsed / 60.0
        ),
        "validation_positions_per_second": float(
            positions / elapsed
        ),
    }
])

display(holdout_summary)

print()
print(
    "NOTE: per_base_ap_hist_1m is a very fine histogram "
    "approximation of raw-float AP."
)
print(
    "per_base_ap_sigmoid_5dp is exact for the concrete "
    "sigmoid + five-decimal quantized transform."
)


NameError: name 'regional_score_chunks' is not defined

In [ ]:
# ============================================================
# SAVE ONE RESULT FILE
# ============================================================

SUMMARY_PATH = (
    RUN_DIR
    / "holdout_summary_10k_SAFE.csv"
)

RESULTS_JSON = (
    RUN_DIR
    / "EXP034_RESULTS_SAFE.json"
)

holdout_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

payload = {
    "experiment": "CNN-EXP-034",
    "step": 10000,
    "evaluation": {
        "mode": "memory_safe_streaming",
        "raw_ap_method": (
            "1,000,001-bin histogram over clamped "
            "log-rate [-20, 15]"
        ),
        "sigmoid_5dp_method": (
            "exact histogram AP after sigmoid(log-rate) "
            "rounded to 5 decimals"
        ),
        "spearman_raw": (
            "exact dense ranks on positive positions only"
        ),
        "regional_metrics": (
            "exact metrics on 32-bp regions"
        ),
    },
    "metrics": (
        holdout_summary
        .where(
            pd.notnull(holdout_summary),
            None,
        )
        .to_dict(orient="records")[0]
    ),
    "reference_EXP033_10k": {
        "regional_ap32": 0.119389,
        "top10_tss_recall": 0.717029,
        "fraction_regions_for_95pct_tss": 0.434711,
    },
}

RESULTS_JSON.write_text(
    json.dumps(
        payload,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print("Saved CSV:", SUMMARY_PATH)
print()
print("Пришли мне этот файл:")
print(RESULTS_JSON)
